In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [7]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from langchain.chat_models import init_chat_model
from typing import Callable

#large_model = init_chat_model("claude-sonnet-4-6")
large_model = init_chat_model("deepseek-v4-pro")
standard_model = init_chat_model("gpt-5-nano")


@wrap_model_call
def state_based_model(request: ModelRequest, 
handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
    """Select model based on State conversation length."""
    # request.messages is a shortcut for request.state["messages"]
    message_count = len(request.messages)  

    if message_count > 10:
        # Long conversation - use model with larger context window
        model = large_model
    else:
        # Short conversation - use efficient model
        model = standard_model

    request = request.override(model=model)  

    return handler(request)

In [8]:
from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5-nano",
    middleware=[state_based_model],
    system_prompt="You are roleplaying a real life helpful office intern."
)

In [9]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [
        HumanMessage(content="Did you water the office plant today?")
        ]}
)

print(response["messages"][-1].content)

I can’t physically water it from here, but I can help make sure it gets taken care of. Here are a few options—which would you prefer?
- Set a daily/alternate-day watering reminder (and send you a quick check-in each time).
- Check with the office/building staff to arrange regular watering.
- If you have a moisture reader or care sheet, I can track moisture levels and suggest the best schedule.

Would you like me to set a reminder for a specific time, or coordinate with someone to water it today?


In [10]:
print(response["messages"][-1].response_metadata["model_name"])

gpt-5-nano-2025-08-07


In [11]:
from langchain.messages import AIMessage

response = agent.invoke(
    {"messages": [
        HumanMessage(content="Did you water the office plant today?"),
        AIMessage(content="Yes, I gave it a light watering this morning."),
        HumanMessage(content="Has it grown much this week?"),
        AIMessage(content="It's sprouted two new leaves since Monday."),
        HumanMessage(content="Are the leaves still turning yellow on the edges?"),
        AIMessage(content="A little, but it's looking healthier overall."),
        HumanMessage(content="Did you remember to rotate the pot toward the window?"),
        AIMessage(content="I rotated it a quarter turn so it gets more even light."),
        HumanMessage(content="How often should we be fertilizing this plant?"),
        AIMessage(content="About once every two weeks with a diluted liquid fertilizer."),
        HumanMessage(content="When should we expect to have to replace the pot?")
        ]}
)

print(response["messages"][-1].content)

Probably not for another year or so, but I can check if it’s getting root-bound. If roots are coming out the drainage holes or it dries out really fast after watering, it’s time.


In [12]:
print(response["messages"][-1].response_metadata["model_name"])

deepseek-v4-pro
